## Dataset & DataLoader
dataset-preprocessing.ipynb 에서 저장한 JSON / tokenizer 를 불러와서 Pytorch Dataset -> DataLoader 까지 구성

#### 0. 임포트

In [51]:
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

In [53]:
print(f"Pytorch: {torch.__version__}")
print(f"MPS 사용 가능: {torch.backends.mps.is_available()}")

Pytorch: 2.5.1
MPS 사용 가능: True


#### 1. 설정

In [56]:
DATA_DIR = Path("data")
MAX_LEN = 128
BATCH_SIZE = 32

PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

#### 2. Tokenizer 로드

In [59]:
tokenizer_file_path = str(DATA_DIR / "tokenizer" / "tokenizer.json")
tokenizer = Tokenizer.from_file(tokenizer_file_path)

VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"Vocab size : {VOCAB_SIZE}")

Vocab size : 16000


#### 3. Json 로드

In [62]:
def load_split(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_split(DATA_DIR / "train.json")
val_data = load_split(DATA_DIR / "val.json")
test_data = load_split(DATA_DIR / "test.json")

print(f"Train 데이터 크기: {len(train_data)}")
print(f"Validation 데이터 크기: {len(val_data)}")
print(f"Test 데이터 크기: {len(test_data)}")

Train 데이터 크기: 69300
Validation 데이터 크기: 8662
Test 데이터 크기: 8663


#### 4. TranslationDataset

In [65]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item["src"], item["tgt"]

In [75]:
train_ds = TranslationDataset(train_data)
val_ds = TranslationDataset(val_data)
test_ds = TranslationDataset(test_data)

# 단건 확인
src, tgt = train_ds[0]
print(f"src len: {len(src)}, tgt len: {len(tgt)}")

src len: 10, tgt len: 17


#### 5. collate_fn 정의 (배치 패딩)

In [89]:
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    def pad(seqs):
        padded = [s[:MAX_LEN] + [PAD_ID] * (MAX_LEN - len(s)) for s in seqs]

        return torch.tensor(padded, dtype=torch.long)

    return pad(src_batch), pad(tgt_batch)

#### 6. DataLoader

In [164]:
train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

#### 7. Batch Shape 확인

In [95]:
src_batch, tgt_batch = next(iter(train_dl))

print(f"src: {src_batch.shape}, dtype={src_batch.dtype}")
print(f"tgt: {tgt_batch.shape}, dtype={tgt_batch.dtype}")

src: torch.Size([32, 128]), dtype=torch.int64
tgt: torch.Size([32, 128]), dtype=torch.int64


#### 8. Paddig mask 생성 (for transformer input)

In [118]:
def make_pad_mask(seq, pad_id=PAD_ID):
    """(B, 1, 1, T) - multi-head attention 브로드캐스팅 용도"""
    return (seq == pad_id).unsqueeze(1).unsqueeze(2)

def mask_casual_mask(seq_len, device="cpu"):
    """(1, 1, T, T) - 디코더 self-attention 인과 마스크 (상삼각 행렬로 구성해서 True 이면 미래 토큰을 의미)"""
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(1)

In [120]:
# 예시
src_batch, tgt_batch = next(iter(train_dl))

example_pad_mask = make_pad_mask(src_batch)
example_casual_mask = mask_casual_mask(MAX_LEN)

print(example_pad_mask.shape)
print(example_casual_mask.shape)

torch.Size([32, 1, 1, 128])
torch.Size([1, 1, 128, 128])


## Transformer 모델 구성

#### 0. 임포트 & 설정

In [124]:
import math, torch, torch.nn as nn, torch.nn.functional as F

In [126]:
VOCAB_SIZE = tokenizer.get_vocab_size()
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 4
D_FF = 1024
DROPOUT = 0.1
MAX_LEN = 128
PAD_ID = 0

In [134]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")

DEVICE = get_device()

print(DEVICE)

mps


#### 1. Positional Embedding

In [181]:
class PositionalEncoding(nn.Module):
    """PE(pos, 2i) = sin(pos / 10000^(2i/d_model))"""
    def __init__(self, d_model=D_MODEL, max_len=MAX_LEN, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        # 분자 값
        pos = torch.arange(max_len).unsqueeze(1).float()
        # 분모 값
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
                * (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe.unsqueeze(0)) # (1, T, D) - Batch 차원을 더해야만 broadcast 가능

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [183]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL):
        super().__init__()
        
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.scale = math.sqrt(d_model)

    # (B, T) -> (B, T, D_MODEL)
    def forward(self, x):
        return self.emb(x) * self.scale

In [193]:
# 임베딩 테스트
src_batch, tgt_batch = next(iter(train_dl))

embed = InputEmbedding()
positional_embed = PositionalEncoding()

input_embed_example = embed(src_batch)
print(input_embed_example.shape)

positional_embed_example = positional_embed(input_embed_example)
print(positional_embed_example.shape)

torch.Size([32, 128, 256])
torch.Size([32, 128, 256])


#### 2. Multi-Head Attention

In [196]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.head_dim = d_model // n_heads
        self.scale = math.sqrt(self.head_dim)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
    
        # 1. Q, K, V projection (B, T, DIM_MODEL) -> (B, T, DIM_MODEL)
        q_proj = self.W_q(q)
        k_proj = self.W_k(k)
        v_proj = self.W_v(v)
    
        # 2. split heads (B, T, HEAD_DIM) -> (B, T, H, HEAD_DIM) -> (B, H, T, HEAD_DIM)
        Q = q_proj.view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        K = k_proj.view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        V = v_proj.view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
    
        # 3. attention score (B, H, T, HEAD_DIM) * (B, H, HEAD_DIM, T) -> (B, H, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
    
        # 4. mask
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
    
        # 5. attention probability
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
    
        # 6. weighted sum (B, H, T, T) * (B, H, T, HEAD_DIM) -> (B, H, T, HEAD_DIM)
        context = torch.matmul(attn, V)
    
        # 7. merge heads (최종적으로 (B, T, DIM_MODEL) 로 만드는게 목표이고, 이 과정에서 해드끼리 병합해야함)
        # (B, H, T, HEAD_DIM) -> (B, T, H, HEAD_DIM)
        # contiguous : 메모리 배치를 view 하기 좋게 재정렬하는 과정
        context = context.transpose(1, 2).contiguous().view(B, -1, self.d_model)
    
        # 8. output projection (B, T, DIM_MODEL)
        output = self.W_o(context)
    
        return output

In [203]:
# MHA 테스트
mha = MultiHeadAttention()

mha_example = mha(positional_embed_example, positional_embed_example, positional_embed_example)
print(mha_example.shape)

torch.Size([32, 128, 256])


#### 3. Feed-Forward Network

In [205]:
class FeedForward(nn.Module):
    def __init__(self, d_model=D_MODEL, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [207]:
# Feed-Forward 테스트
ff = FeedForward()

ff_example = ff(mha_example)
print(ff_example.shape)

torch.Size([32, 128, 256])


#### 4. Encoder Layer

In [210]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: torch.Tensor = None) -> torch.Tensor:
        # Self-Attention + residual
        x = self.self_attn(x, x, x, src_mask)
        x = x + self.dropout(x)
        x = self.norm1(x)

        # FFN + Resu
        x = self.ff(x)
        x = x + self.dropout(x)
        x = self.norm2(x)

        return x

In [212]:
# Encoder 동작 확인
enc_layer = EncoderLayer()
enc_layer_example = enc_layer(positional_embed_example)

print(enc_layer_example.shape)

torch.Size([32, 128, 256])


#### 5. Decoder Layer